# Multi-Modal DistilBERT Yelp Fake Detector — CONTRASTIVE (SupCon, tuned, 642k, no AI)

Ablation variant of Model B's DistilBERT member: same text+behavioural fusion, but the DL
member is trained with a **supervised contrastive** auxiliary objective (Khosla 2020) jointly
with cross-entropy (Gunel 2021). **The standard Model B is untouched** — a separate detector for
the focal-vs-contrastive-vs-standard comparison.

* **Text branch:** DistilBERT → [CLS] (768-d) → 256-d
* **Behavioural branch:** MLP over the 17 reviewer/business features → 32-d
* **Fusion:** concat(text, behavioural) → head → 1 logit (the served path, exported to ONNX)
* **Contrastive head (train only):** fused → proj → 128-d on the unit sphere; SupCon pulls
  same-label reps together / pushes different-label apart. **Not exported** — serving is a
  drop-in for standard Model B.
* **Loss:** `(1-λ)·CE + λ·SupCon` on balanced 50/50 batches (so SupCon always sees both classes;
  train is the true ~11% fake). Early stopping on val macro-F1.

**Best-shot / tuned:** λ (the dominant CE↔SupCon mix) **and** τ (temperature) are **swept** —
**Phase A** tunes (λ, τ) on a fast proxy (an 80k stratified subsample, balanced batches, 2
epochs, scored on the full balanced val set), then **Phase B** retrains from scratch at the best
config on the **full** 642k train. The chosen (λ, τ) are recorded in the meta.

**Expectation (honest):** the text-only contrastive ablation landed *below* baseline, and SupCon
only reshapes the embedding geometry (adds no new signal beyond the fused features), so even
tuned this may not beat standard Model B. The sweep gives it a genuine best shot.

**Before running:** upload `data/yelp_multimodal_features_full.csv` (~480 MB) to Drive root;
**GPU** runtime (A100/L4; high-RAM advised). **Output:** downloads
`yelp_multimodal_distilbert_contrastive_onnx.zip` (ONNX + scaler + meta + tokenizer/) → extract
to `data/yelp_multimodal_distilbert_contrastive_onnx/`. Feature order matches `model_b.FEAT_COLS`.
**Reference (same Yelp test):** text-only ~69% · standard Model B (full) ens 76.2% / xgb 77.3% / DL ~76.7%.

In [ ]:
!pip install -q -U transformers

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizerFast
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

CSV='/content/drive/MyDrive/yelp_multimodal_features_full.csv'   # PURE full 678k (no AI)
MAX_LEN=256; BATCH=64; EPOCHS=4; LR=2e-5     # BATCH 64 to cut steps on 642k; drop to 32 if OOM
DROP={'user_reviews_on_this_biz'}
META={'text','label','split','source','fake_type','generator','length_bucket'}
device='cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

df=pd.read_csv(CSV); df['text']=df['text'].astype(str)
feat_cols=[c for c in df.columns if c not in META and c not in DROP]
print('rows', len(df), '| behavioural features', len(feat_cols))
tr=df[df.split=='train'].reset_index(drop=True)
va=df[df.split=='val'].reset_index(drop=True)
te=df[df.split=='test'].reset_index(drop=True)

sc=StandardScaler()
Btr=sc.fit_transform(tr[feat_cols].values).astype('float32')
Bva=sc.transform(va[feat_cols].values).astype('float32')
Bte=sc.transform(te[feat_cols].values).astype('float32')

# CLASS IMBALANCE (pure full data): train is the true ~11% fake (val/test balanced). No AI ×10 —
# instead per-sample weight = n_neg/n_pos for the minority (== BCE pos_weight); val-tuned thr sets
# the final operating point.
POS_WEIGHT=float((tr.label.values==0).sum())/max(int((tr.label.values==1).sum()),1)
sw_tr=np.where(tr.label.values==1, POS_WEIGHT, 1.0).astype('float32')
print('train fake rate', round(float(tr.label.mean()),3), '| POS_WEIGHT', round(POS_WEIGHT,2))

In [ ]:
tok=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class MMData(Dataset):
    def __init__(self, texts, beh, labels, weights=None):
        self.enc=tok(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        self.beh=torch.tensor(beh); self.y=torch.tensor(np.asarray(labels), dtype=torch.float32)
        self.w=torch.tensor(weights if weights is not None else np.ones(len(labels), dtype='float32'))
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (self.enc['input_ids'][i], self.enc['attention_mask'][i],
                self.beh[i], self.y[i], self.w[i])

# Balanced batch sampler: each train batch is 50/50 genuine/fake so SupCon always sees positives
# AND negatives of both classes (train is the true ~11% fake, so random batches would be nearly
# all genuine and the contrastive term would degenerate). Minority (fake) indices are resampled
# with replacement across an epoch; one epoch == enough batches to cover the majority class once.
class BalancedBatchSampler(torch.utils.data.Sampler):
    def __init__(self, labels, batch_size):
        self.labels=np.asarray(labels).astype(int); self.bs=batch_size; self.half=batch_size//2
        self.pos=np.where(self.labels==1)[0]; self.neg=np.where(self.labels==0)[0]
        self.n_batches=len(self.neg)//self.half          # cover the majority class once per epoch
    def __iter__(self):
        neg=np.random.permutation(self.neg)
        pos=np.random.choice(self.pos, size=self.n_batches*self.half, replace=True)
        for b in range(self.n_batches):
            sl=slice(b*self.half,(b+1)*self.half)
            idx=np.concatenate([neg[sl], pos[sl]]); np.random.shuffle(idx)
            yield idx.tolist()
    def __len__(self): return self.n_batches

tr_ds=MMData(tr.text.values, Btr, tr.label.values, sw_tr)
tr_dl=DataLoader(tr_ds, batch_sampler=BalancedBatchSampler(tr.label.values, BATCH))
va_dl=DataLoader(MMData(va.text.values, Bva, va.label.values), batch_size=64)
te_dl=DataLoader(MMData(te.text.values, Bte, te.label.values), batch_size=64)
print('balanced train batches/epoch:', len(tr_dl), '| batch', BATCH, '(50/50 genuine/fake)')

In [ ]:
import torch.nn.functional as F

class MultiModalDistilBert(nn.Module):
    def __init__(self, n_beh):
        super().__init__()
        self.bert=DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.text_proj=nn.Sequential(nn.Linear(768,256), nn.ReLU(), nn.Dropout(0.3))
        self.beh=nn.Sequential(nn.Linear(n_beh,64), nn.ReLU(), nn.Dropout(0.3),
                               nn.Linear(64,32), nn.ReLU())
        self.head=nn.Sequential(nn.Linear(256+32,64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64,1))
        # SupCon projection head on the fused representation -> 128-d unit sphere (contrastive
        # term only; NOT exported to ONNX — serving uses forward()/head only).
        self.proj=nn.Sequential(nn.Linear(256+32,128), nn.ReLU(), nn.Linear(128,128))
    def _fused(self, ids, mask, beh):
        cls=self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:,0]   # [CLS]
        return torch.cat([self.text_proj(cls), self.beh(beh)], dim=1)
    def forward(self, ids, mask, beh):                       # ONNX/serving path: logit only
        return self.head(self._fused(ids, mask, beh)).squeeze(-1)
    def forward_joint(self, ids, mask, beh):                 # train path: logit + L2-normed embedding
        fused=self._fused(ids, mask, beh)
        return self.head(fused).squeeze(-1), F.normalize(self.proj(fused), dim=1)

# Layer-wise LR decay (LLRD): deeper DistilBERT layers + new heads train faster,
# embeddings slowest — the project's standard fine-tuning recipe. proj head trains at head_lr.
def make_optimizer(m, base_lr=LR, head_lr=1e-3, decay=0.95, wd=0.01):
    b=m.bert; n=len(b.transformer.layer); g=[]
    g.append({'params':list(b.embeddings.parameters()), 'lr':base_lr*(decay**n)})
    for i,layer in enumerate(b.transformer.layer):
        g.append({'params':list(layer.parameters()), 'lr':base_lr*(decay**(n-1-i))})
    g.append({'params':list(m.text_proj.parameters())+list(m.beh.parameters())
                      +list(m.head.parameters())+list(m.proj.parameters()),
              'lr':head_lr})
    return torch.optim.AdamW(g, weight_decay=wd)

# Supervised contrastive loss (Khosla 2020) on the L2-normed fused embeddings. Joint with CE
# (Gunel 2021): loss = (1-lambda)*CE + lambda*SupCon. lambda AND temperature are SWEPT (cell-5).
def supcon_loss(emb, labels, temperature):
    dev=emb.device; B=emb.shape[0]
    sim=torch.matmul(emb, emb.T) / temperature
    sim=sim - sim.max(dim=1, keepdim=True)[0].detach()           # numerical stability
    self_mask=torch.eye(B, dtype=torch.bool, device=dev)
    labels=labels.view(-1,1)
    pos_mask=(labels==labels.T) & ~self_mask                     # same-label, excl. self
    exp_sim=torch.exp(sim).masked_fill(self_mask, 0.0)
    log_prob=sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)
    pos_cnt=pos_mask.sum(dim=1)
    mean_log_prob=(pos_mask*log_prob).sum(dim=1) / pos_cnt.clamp(min=1)
    valid=pos_cnt>0                                              # only anchors with >=1 positive
    return -(mean_log_prob[valid].mean()) if valid.any() else torch.zeros((), device=dev)

# ── reusable train/eval helpers (fresh model per call; sweep + final share code) ──
USE_AMP = (device == 'cuda')
loss_fn = nn.BCEWithLogitsLoss(reduction='none')

def predict(m, dl):
    m.eval(); probs=[]
    with torch.no_grad():
        for ids,mask,beh,y,w in dl:
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logit=m(ids.to(device), mask.to(device), beh.to(device))
            else:
                logit=m(ids.to(device), mask.to(device), beh.to(device))
            probs.append(torch.sigmoid(logit.float()).cpu().numpy())
    return np.concatenate(probs)

def best_thr(y, p):
    bf,bt=-1,0.5
    for t in np.linspace(0.05,0.95,19):
        s=f1_score(y,(p>=t).astype(int),average='macro')
        if s>bf: bf,bt=s,t
    return bt,bf

def fit_once(lam, temp, train_dl, epochs, patience=3, tag=''):
    """Train a fresh model with joint (1-lam)*CE + lam*SupCon(temp) on balanced batches;
    early-stop on val macro-F1; restore best. Returns (model, best_thr, best_f1, trh, vah)."""
    m=MultiModalDistilBert(len(feat_cols)).to(device)
    opt=make_optimizer(m)
    total=len(train_dl)*epochs
    sched=get_linear_schedule_with_warmup(opt, int(0.1*total), total)   # 10% warmup
    sc_amp=torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best_f1, best_state, best_t, wait = -1, None, 0.5, 0
    trh, vah = [], []
    for ep in range(epochs):
        m.train(); tot=tot_ce=tot_sc=0
        pbar=tqdm(train_dl, desc=f'{tag} ep{ep+1}/{epochs}', leave=False)
        for ids,mask,beh,y,w in pbar:
            ids,mask,beh,y=ids.to(device),mask.to(device),beh.to(device),y.to(device)
            opt.zero_grad()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logit,emb=m.forward_joint(ids, mask, beh)
                    ce=loss_fn(logit, y).mean()
                sc_loss=supcon_loss(emb.float(), y, temp)        # fp32 contrastive term
                loss=(1-lam)*ce + lam*sc_loss
                sc_amp.scale(loss).backward(); sc_amp.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                sc_amp.step(opt); sc_amp.update()
            else:
                logit,emb=m.forward_joint(ids, mask, beh)
                ce=loss_fn(logit, y).mean()
                sc_loss=supcon_loss(emb.float(), y, temp)
                loss=(1-lam)*ce + lam*sc_loss
                loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            sched.step(); tot+=loss.item(); tot_ce+=ce.item(); tot_sc+=float(sc_loss)
            pbar.set_postfix(ce=f'{ce.item():.3f}', sup=f'{float(sc_loss):.3f}')
        p_va=predict(m, va_dl); thr,f1=best_thr(va.label.values, p_va)
        trh.append(tot/len(train_dl)); vah.append(f1)
        print(f'  {tag} ep{ep+1}: ce={tot_ce/len(train_dl):.4f} sup={tot_sc/len(train_dl):.4f}  val_macroF1={f1:.4f} @thr {thr:.2f}')
        if f1>best_f1:
            best_f1, best_t = f1, thr
            best_state={k:v.cpu().clone() for k,v in m.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: print('   early stop'); break
    m.load_state_dict(best_state)
    return m, best_t, best_f1, trh, vah

print('helpers ready (model + supcon + fit_once)')

In [ ]:
import itertools
# ============================ PHASE A — hyperparameter sweep ============================
# Tune the contrastive mix (lambda) and temperature (tau) on a fast PROXY: a stratified train
# subsample with balanced 50/50 batches + few epochs, scored on the FULL balanced val set.
# lambda is the dominant knob (how much SupCon vs CE); tau shapes the similarity sharpness.
PROXY_N      = 80_000
PROXY_EPOCHS = 2
LAMBDAS = [0.1, 0.3, 0.5]          # contrastive weight (0 == plain balanced-batch CE)
TEMPS   = [0.1, 0.3]               # SupCon temperature

rng = np.random.default_rng(42)
idx = rng.choice(len(tr), size=min(PROXY_N, len(tr)), replace=False)
proxy_ds = MMData(tr.text.values[idx], Btr[idx], tr.label.values[idx], sw_tr[idx])
proxy_dl = DataLoader(proxy_ds, batch_sampler=BalancedBatchSampler(tr.label.values[idx], BATCH))
print(f'PROXY sweep: {len(idx)} rows (fake rate {tr.label.values[idx].mean():.3f}, balanced batches) | '
      f'{len(LAMBDAS)*len(TEMPS)} configs x {PROXY_EPOCHS} epochs\n')

sweep=[]
for lam,temp in itertools.product(LAMBDAS, TEMPS):
    _, _, f1, _, _ = fit_once(lam, temp, proxy_dl, PROXY_EPOCHS, patience=2, tag=f'[L{lam} t{temp}]')
    sweep.append((f1, lam, temp))
    print(f'  config lambda={lam} tau={temp}  ->  proxy val macroF1={f1:.4f}\n')
sweep.sort(reverse=True)
print('PROXY ranking (best first):')
for f1,lam,temp in sweep: print(f'   lambda={lam} tau={temp}: {f1:.4f}')
BEST_F1, BEST_LAMBDA, BEST_TEMP = sweep[0]
print(f'\n==> BEST proxy config: lambda={BEST_LAMBDA}, tau={BEST_TEMP}  (proxy val {BEST_F1:.4f})')

# ============================ PHASE B — final full-data train ============================
print('\n=============== FINAL: full-data train @ best config ===============')
model, best_thr_v, final_f1, tr_hist, val_hist = fit_once(
    BEST_LAMBDA, BEST_TEMP, tr_dl, EPOCHS, patience=3, tag='FINAL')
SUPCON_LAMBDA, SUPCON_TEMP = BEST_LAMBDA, BEST_TEMP   # recorded into meta on export
print(f'FINAL full-data val macroF1={final_f1:.4f} @thr {best_thr_v:.2f} '
      f'(lambda={SUPCON_LAMBDA}, tau={SUPCON_TEMP})')

In [ ]:
# Training curves
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,len(tr_hist)+1),tr_hist,marker='o'); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(range(1,len(val_hist)+1),val_hist,marker='o',color='green'); ax[1].set_title('Val macro-F1'); ax[1].set_xlabel('epoch')
ax[1].axvline(int(np.argmax(val_hist))+1, ls='--', color='red', label='best epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('yelp_multimodal_distilbert_training.png', dpi=120); plt.show()

In [ ]:
p_te=predict(model, te_dl); pred=(p_te>=best_thr_v).astype(int); y_te=te.label.values
print(f'best contrastive config: lambda={SUPCON_LAMBDA}, tau={SUPCON_TEMP}')
print('threshold =', round(best_thr_v,2))
print('test accuracy  =', round(accuracy_score(y_te,pred),4))
print('test macro-F1  =', round(f1_score(y_te,pred,average='macro'),4))
print('fake recall    =', round(recall_score(y_te,pred,pos_label=1),3))
print('fake precision =', round(precision_score(y_te,pred,pos_label=1),3))
print('genuine spec   =', round(recall_score(y_te,pred,pos_label=0),3))
print('\nReferences (same Yelp test): text-only ~69% · pure sklearn Model B (full) ens 76.2% / xgb 77.3% · standard Model B DL ~76.7%')
print()
print(classification_report(y_te, pred, target_names=['genuine','fake'], digits=4))

In [ ]:
# Export to ONNX (thread-safe serving) + bundle scaler/threshold/tokenizer.
# NOTE: only forward() (logit) is exported — the SupCon proj head is training-only, so serving
# is identical to standard Model B (drop-in: same inputs/outputs).
!pip install -q onnx onnxruntime onnxscript
import os, json, joblib, torch, zipfile, numpy as np
model.eval()
dummy_ids =torch.zeros(1,MAX_LEN,dtype=torch.long,device=device)
dummy_mask=torch.ones(1,MAX_LEN,dtype=torch.long,device=device)
dummy_beh =torch.zeros(1,len(feat_cols),dtype=torch.float32,device=device)
_args=(dummy_ids,dummy_mask,dummy_beh)
_kw=dict(input_names=['input_ids','attention_mask','beh'], output_names=['logit'],
         dynamic_axes={'input_ids':{0:'b'},'attention_mask':{0:'b'},'beh':{0:'b'},'logit':{0:'b'}},
         opset_version=14, do_constant_folding=True)
try:
    torch.onnx.export(model,_args,'yelp_multimodal_distilbert.onnx',dynamo=False,**_kw)  # stable legacy exporter
except TypeError:
    torch.onnx.export(model,_args,'yelp_multimodal_distilbert.onnx',**_kw)               # older torch: legacy is default
joblib.dump(sc,'yelp_multimodal_distilbert_scaler.joblib')
tok.save_pretrained('yelp_multimodal_distilbert_tok')   # HF tokenizer = a folder of files
json.dump({'threshold':float(best_thr_v),'feat_cols':feat_cols,'max_len':MAX_LEN,
           'supcon_lambda':float(SUPCON_LAMBDA),'supcon_tau':float(SUPCON_TEMP)},
          open('yelp_multimodal_distilbert_meta.json','w'), indent=2)

# sanity check: ONNX vs PyTorch on 4 test rows
import onnxruntime as ort
sess=ort.InferenceSession('yelp_multimodal_distilbert.onnx',providers=['CPUExecutionProvider'])
enc=tok(list(te.text.values[:4]),truncation=True,padding='max_length',max_length=MAX_LEN,return_tensors='np')
onnx_logit=sess.run(None,{'input_ids':enc['input_ids'].astype('int64'),
                          'attention_mask':enc['attention_mask'].astype('int64'),
                          'beh':Bte[:4]})[0].flatten()
print('ONNX inputs:', [i.name for i in sess.get_inputs()])
print('ONNX prob :', np.round(1/(1+np.exp(-onnx_logit)),4))
print('Torch prob:', np.round(p_te[:4],4))

with zipfile.ZipFile('yelp_multimodal_distilbert_contrastive_onnx.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['yelp_multimodal_distilbert.onnx','yelp_multimodal_distilbert_scaler.joblib',
               'yelp_multimodal_distilbert_meta.json']:
        zf.write(fn)
    for f in os.listdir('yelp_multimodal_distilbert_tok'):   # bundle the tokenizer folder INSIDE the one zip
        zf.write(os.path.join('yelp_multimodal_distilbert_tok', f), arcname=f'tokenizer/{f}')
from google.colab import files
files.download('yelp_multimodal_distilbert_contrastive_onnx.zip')  # extract -> data/yelp_multimodal_distilbert_contrastive_onnx/
print('done: one zip -> onnx + scaler + meta + tokenizer/  (extract -> data/yelp_multimodal_distilbert_contrastive_onnx/)')